# Email Spam Classifier - EDA and Preprocessing

## Project Overview
This notebook performs Exploratory Data Analysis (EDA) and text preprocessing on the SMS Spam Collection dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import re

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

## 1. Load Dataset

Dataset: SMS Spam Collection Dataset
Source: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [ ]:
# Load dataset
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Keep only relevant columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Convert labels to binary
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Basic Statistics

In [ ]:
# Dataset info
print("Dataset Info:")
print(df.info())

# Check for missing values
print(f"\nMissing Values:\n{df.isnull().sum()}")

# Label distribution
print(f"\nLabel Distribution:\n{df['label'].value_counts()}")

## 3. Visualize Label Distribution

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
label_counts = df['label'].value_counts()
axes[0].bar(['Ham (0)', 'Spam (1)'], label_counts.values, color=['green', 'red'])
axes[0].set_title('Distribution of Spam vs Ham')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(label_counts.values, labels=['Ham', 'Spam'], autopct='%1.1f%%', colors=['green', 'red'])
axes[1].set_title('Spam vs Ham Percentage')

plt.tight_layout()
plt.show()

## 4. Text Analysis

In [ ]:
# Add text features
df['message_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))
df['uppercase_count'] = df['message'].apply(lambda x: sum(1 for c in x if c.isupper()))
df['exclamation_count'] = df['message'].apply(lambda x: x.count('!'))

# Display statistics
print("Text Features Statistics:")
df.groupby('label')[['message_length', 'word_count', 'uppercase_count', 'exclamation_count']].describe()

## 5. Visualize Text Features

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Message length distribution
for label in [0, 1]:
    subset = df[df['label'] == label]
    axes[0, 0].hist(subset['message_length'], alpha=0.5, label='Ham' if label == 0 else 'Spam', bins=30)
axes[0, 0].set_title('Message Length Distribution')
axes[0, 0].set_xlabel('Message Length')
axes[0, 0].legend()

# Word count distribution
for label in [0, 1]:
    subset = df[df['label'] == label]
    axes[0, 1].hist(subset['word_count'], alpha=0.5, label='Ham' if label == 0 else 'Spam', bins=30)
axes[0, 1].set_title('Word Count Distribution')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].legend()

# Uppercase count distribution
for label in [0, 1]:
    subset = df[df['label'] == label]
    axes[1, 0].hist(subset['uppercase_count'], alpha=0.5, label='Ham' if label == 0 else 'Spam', bins=30)
axes[1, 0].set_title('Uppercase Count Distribution')
axes[1, 0].set_xlabel('Uppercase Count')
axes[1, 0].legend()

# Exclamation count distribution
for label in [0, 1]:
    subset = df[df['label'] == label]
    axes[1, 1].hist(subset['exclamation_count'], alpha=0.5, label='Ham' if label == 0 else 'Spam', bins=30)
axes[1, 1].set_title('Exclamation Count Distribution')
axes[1, 1].set_xlabel('Exclamation Count')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 6. Word Cloud Analysis

In [ ]:
# Create word clouds for spam and ham
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Spam word cloud
spam_text = ' '.join(df[df['label'] == 1]['message'])
spam_wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(spam_text)
axes[0].imshow(spam_wordcloud, interpolation='bilinear')
axes[0].set_title('Spam Messages - Word Cloud', fontsize=14)
axes[0].axis('off')

# Ham word cloud
ham_text = ' '.join(df[df['label'] == 0]['message'])
ham_wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(ham_text)
axes[1].imshow(ham_wordcloud, interpolation='bilinear')
axes[1].set_title('Ham Messages - Word Cloud', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 7. Text Preprocessing

In [ ]:
import sys
sys.path.append('..')
from src.text_preprocessing import TextPreprocessor

# Initialize preprocessor
preprocessor = TextPreprocessor(method='stemming')

# Apply preprocessing
df['cleaned_message'] = df['message'].apply(preprocessor.preprocess)

# Display examples
print("Original vs Cleaned Messages:")
print("=" * 80)
for i in range(5):
    print(f"Original: {df['message'].iloc[i]}")
    print(f"Cleaned:  {df['cleaned_message'].iloc[i]}")
    print("-" * 80)

## 8. Save Preprocessed Data

In [ ]:
# Save preprocessed data
df.to_csv('../data/preprocessed_spam.csv', index=False)
print("Preprocessed data saved to '../data/preprocessed_spam.csv'")